<img src="https://raw.githubusercontent.com/reflective-org/AAAR-2026-Aerosol-Tutorial/9feba6889d498d8c754a6783e5460f46e054d60f/assets/banner-inside.png" width="100%" alt="AAAR 2026 photochemical box modeling tutorial. Inside the model (optional, after Level 3). One coupling interval, written out step by step. Supported by Reflective.">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/reflective-org/AAAR-2026-Aerosol-Tutorial/blob/main/notebooks/04_inside_the_model.ipynb)

The other notebooks and the web app treat the box model as a black box: you change inputs and
look at outputs. This notebook opens the box. With the same `tomas_jax` building blocks that
`run()` calls, you will (1) compute photolysis frequencies with TUV-x and compare them with the
MCM formula, (2) turn the MCM limonene mechanism into a set of ODEs and solve one interval with
the stiff solver, (3) check the model's gas–particle partitioning against the Pankow equation,
(4) get the N₂O₅ uptake rate from γ and the particle surface area, and (5) write out the loop
that couples gas chemistry and aerosol, then check it against the model.

> <font color="#8B8D8E"><b>HOW TO USE IT</b></font> &nbsp; Run the cells from top to bottom (Colab: *Runtime › Run all*). Boxes labelled <font color="#A47F52"><b>TRY THIS</b></font> name a variable to change: edit it, then run that cell again and the cells below it in the same section. Where the value is a simple number, text or list, Colab also shows a title and an input box; editing the box changes the code. Sections build on each other (Section 5, for example, reuses the mechanism from Section 2), so after a restart run from the top. All plots are interactive: hover for values, drag to zoom, double-click to reset, click a legend entry to hide or show a line. The *Size resolution* cell below sets how many size bins every run uses.

<details>
<summary><b>Run times</b></summary>

On a laptop (Colab is about 2× slower): setup a few seconds (plus 1–2 min to install on Colab the first time); Sections 1–4 a few seconds each; Section 5 runs the 24 h Level 3 case once through a hand-written copy of the model loop (about 15 s). The first call of each JAX function compiles it, so the first run of a cell is slower than a re-run. A precomputed Level 3 result at the start of Section 5 needs no model run at all. The whole notebook runs in about 30 s on a laptop.

</details>

## Learning goals

> <font color="#8B8D8E"><b>LEARNING GOALS</b></font> &nbsp; After this notebook you can:
>
> 1. Say where photolysis frequencies come from, and how TUV-x and the MCM parameterisation differ.
> 2. Read a KPP/MCM reaction line and explain how it becomes part of $d\mathbf{y}/dt$, what the RO₂ sum is, and why the system is stiff.
> 3. Write down the equilibrium particle fraction $F = (1 + c^*/C_{OA})^{-1}$ and check that the kinetic partitioning code relaxes to it.
> 4. Compute $k_\text{het} = \gamma \bar c A / 4$ and say when that simple formula holds.
> 5. Describe the operator-splitting loop and what is held fixed during one coupling interval.

## Setup

> <font color="#8B8D8E"><b>SETUP</b></font> &nbsp; Run the next two cells once per session.
>
> - **Colab:** the first cell installs the box model and the tutorial helpers (1–2 minutes). Red pip
>   "dependency conflicts" lines are expected. If a cell asks you to restart, click
>   *Runtime › Restart session*, then *Runtime › Run all*.
> - **Your own computer:** install the repository first ([README](https://github.com/reflective-org/AAAR-2026-Aerosol-Tutorial/blob/main/README.md#run-on-your-own-computer));
>   the first cell then does nothing.
> - `at.setup()` downloads the TUV-x photolysis data once (about 5 MB) and prints the versions.

In [ ]:
# Colab: installs the tutorial helpers and the box model (first time: 1-2 minutes).
# Red pip "dependency conflicts" lines on Colab are expected and do not matter here.
import importlib.metadata
import sys

HELPERS_VERSION = "0.4.5"   # the aaar_tutorial version these notebooks need


def _older(version):
    return tuple(map(int, version.split("."))) < tuple(map(int, HELPERS_VERSION.split(".")))


try:
    installed = importlib.metadata.version("aaar-tutorial")
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed is None or _older(installed):
    if "google.colab" not in sys.modules:
        raise ModuleNotFoundError(
            f"This notebook needs the tutorial helpers {HELPERS_VERSION}, found {installed or 'none'}. Pick the "
            "kernel of the repository's .venv, update it (git pull, then uv sync or pip install -e .), then "
            "restart the kernel.")
    %pip install -q "aaar-tutorial @ git+https://github.com/reflective-org/AAAR-2026-Aerosol-Tutorial@main"
    importlib.invalidate_caches()           # so the next cell finds the new packages
loaded = getattr(sys.modules.get("aaar_tutorial"), "__version__", None)
if loaded is not None and _older(loaded):
    raise RuntimeError("This session still has older tutorial helpers loaded. Colab: Runtime > Restart "
                       "session, then Runtime > Run all. Locally: Kernel > Restart Kernel.")

In [ ]:
import aaar_tutorial as at
from aaar_tutorial import config, plots

at.setup()

In [ ]:
#@title Size resolution
# Size bins for every run in this notebook. 40 (the web app's grid) is the fastest; 80, 160 and 320
# draw the size distribution more finely (results change by under 1 %) and take longer: a Level 1
# run takes about 1, 4, 14 and 60 s on a laptop, Level 3 about 12, 19, 39 and 140 s; Colab is
# about twice as slow. Rerun the cells below after changing it.
N_BINS = 40  #@param [40, 80, 160, 320] {type:"raw"}
at.use_bins(N_BINS)

In [ ]:
import time
from datetime import datetime

import diffrax as dfx
import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from tomas_jax.box.config import AerosolStepConfig

COLORS = config.SERIES_COLORS
BOXVOL = AerosolStepConfig().boxvol_cm3   # 1e10 cm3: TOMAS stores number and mass per box ("per cell")
# plots.style(fig, title) gives the figures below the look of the aaar_tutorial plots

## 1. Photolysis: where J comes from

A photolysis frequency $J$ (s⁻¹) is the first-order rate constant of a reaction such as
NO₂ + hν → NO + O:

$$J = \int \sigma(\lambda, T)\, \phi(\lambda, T)\, F(\lambda)\, d\lambda$$

with $\sigma$ the absorption cross section, $\phi$ the quantum yield and $F$ the actinic flux
(photons arriving from all directions). The model can get $J$ in two ways:

- **TUV-x** ([tuvx-jax](https://github.com/reflective-org/tuvx-jax), a Python port of NCAR's
  TUV-x) solves radiative transfer through a standard atmosphere (clear sky, no aerosol, surface
  albedo 0.1) and evaluates the integral. At the surface $J$ then depends only on the solar
  zenith angle $\chi$ and the Earth–Sun distance, so the model builds a $J(\chi)$ table once per
  session (1° grid) and interpolates in it.
- **MCM parameterisation** ([Saunders et al., 2003](https://doi.org/10.5194/acp-3-161-2003)): a fit to clear-sky surface values with three
  constants per reaction,
  $$J = l\,(\cos\chi)^m \exp(-n \sec\chi).$$

Levels 2 and 3 use TUV-x (Level 2 rescales it to your J(NO₂) and keeps it constant). Of the 28
$J$ values in the limonene mechanism, 25 come from TUV-x and 3 from the MCM formula because TUV-x
has no recipe for them (ASSUMPTIONS A7).

The cell below calls TUV-x the way the model does when it builds its table (about 2 s).

> <font color="#A47F52"><b>TRY THIS</b></font> &nbsp; Add `"J_NO3_NO2"` or `"J_HONO"` to `REACTIONS`. The cell prints every name you can use.

In [ ]:
#@title Choose the reactions and compute J with TUV-x
from tomas_jax.gaschem import mcm_rates, photolysis
from tomas_jax.gaschem.config import TUVX_CONFIG
from tuvx_photolysis import PhotolysisCalculator

# MCM photolysis reactions (Try this: add "J_NO3_NO2" or "J_HONO")
REACTIONS = ["J_NO2", "J_O3_O1D", "J_HCHO_H", "J_H2O2"]  #@param {type:"raw"}
SZA_DEG = np.arange(0.0, 90.0, 2.0)

root = photolysis.tuvx_root()                              # the TUV-x data that setup() downloaded
tuvx = PhotolysisCalculator.from_tuvx_json(root / TUVX_CONFIG, data_root=root)
j_tuvx = {name: [] for name in REACTIONS}
for sza in SZA_DEG:
    column = tuvx.rate_constants_profile(float(sza), 1.0)  # J at every altitude level, Earth-Sun distance 1 AU
    for name in REACTIONS:
        key = photolysis.TUVX_DIRECT[name]                 # MCM name -> TUV-x name, e.g. "NO2+hv->NO+O(3P)"
        j_tuvx[name].append(np.interp(0.0, tuvx.height_edges_km, column[key]))   # surface value
j_tuvx = {name: np.array(values) for name, values in j_tuvx.items()}
j_mcm = mcm_rates.mcm_photolysis(np.deg2rad(SZA_DEG), REACTIONS)

print("Names you can use in REACTIONS:", ", ".join(sorted(photolysis.TUVX_DIRECT)))
for name in REACTIONS:
    print(f"{name:10s} MCM constants (l, m, n) = {mcm_rates.MCM_J_PARAMS[name]}")

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.65, 0.35], vertical_spacing=0.06)
for n, name in enumerate(REACTIONS):
    color = COLORS[n % len(COLORS)]
    fig.add_trace(go.Scatter(x=SZA_DEG, y=j_tuvx[name], name=f"{name} TUV-x", legendgroup=name,
                             line=dict(color=color)), row=1, col=1)
    fig.add_trace(go.Scatter(x=SZA_DEG, y=j_mcm[name], name=f"{name} MCM", legendgroup=name,
                             line=dict(color=color, dash="dash")), row=1, col=1)
    fig.add_trace(go.Scatter(x=SZA_DEG, y=j_tuvx[name] / j_mcm[name], name=name, legendgroup=name,
                             showlegend=False, line=dict(color=color)), row=2, col=1)
fig.update_yaxes(type="log", title="J (s⁻¹)", range=[-8, -1.5], exponentformat="power", row=1, col=1)
fig.update_yaxes(title="TUV-x / MCM", range=[0.5, 3], row=2, col=1)   # near 90° the ratio runs off the scale
fig.update_xaxes(title="Solar zenith angle (°)", row=2, col=1)
plots.style(fig, "Surface J: TUV-x (solid) and the MCM formula (dashed)", height=620, legend_right=True)

> <font color="#858F6A"><b>WHAT TO NOTICE</b></font> &nbsp; All $J$ fall as the sun gets lower, but J(O¹D) falls fastest: it needs UV below about 330 nm, which ozone absorbs strongly along a long slant path. With the sun high (SZA below 60°), TUV-x gives 10–35 % more than the MCM fit for these reactions. With the sun low the gap grows, most for J(O¹D): about 2× at 76° and 4× at 80°. Close to 90° both values are tiny and the ratio runs off the scale (zoom out to see it). The two use different atmospheres and cross-section data. Differences of this size are one reason why box models disagree.

### J over a day

During a run the driver asks a `JProvider` for all 28 $J$ values at the middle of each
coupling interval. It turns the time into a solar zenith angle for the site and date and
looks it up in the table. Below are both modes for the Level 3 site (Buffalo, NY) on 1 July.
The first call builds the 1° table (about 3 s).

> <font color="#A47F52"><b>TRY THIS</b></font> &nbsp; Set `LATITUDE = 61.85` and `LONGITUDE = 24.28` (Hyytiälä, Finland) with `START_UTC = "2026-06-30T21:00:00+00:00"`, or keep Buffalo and set `START_UTC = "2026-01-01T05:00:00+00:00"`.

In [ ]:
#@title Choose the site and start time
from tomas_jax.gaschem.photolysis import JProvider, PhotolysisSettings

level3 = at.scenario_from("level3_day_night")
LATITUDE = level3.photolysis.latitude_deg      # Try this: 61.85 (Hyytiälä, Finland)
LONGITUDE = level3.photolysis.longitude_deg    # Try this: 24.28 (and START_UTC = "2026-06-30T21:00:00+00:00")
# local midnight in Buffalo (EDT = UTC-4); Try this: "2026-01-01T05:00:00+00:00"
START_UTC = "2026-07-01T04:00:00+00:00"  #@param {type:"string"}

hours = np.arange(0.0, 24.01, 0.25)
j_day = {}
for mode in ("tuvx", "mcm"):
    light = JProvider(REACTIONS, PhotolysisSettings(mode=mode, latitude=LATITUDE, longitude=LONGITUDE,
                                                    start_utc=datetime.fromisoformat(START_UTC)))
    j_day[mode] = np.array([light.at(h * 3600.0) for h in hours])        # [time, reaction]
sza_day = np.array([light.sza_deg(h * 3600.0) for h in hours])

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.65, 0.35], vertical_spacing=0.06)
for mode, dash in (("tuvx", None), ("mcm", "dash")):
    fig.add_trace(go.Scatter(x=hours, y=j_day[mode][:, 0], name=f"{REACTIONS[0]} {mode}",
                             line=dict(color=COLORS[0], dash=dash)), row=1, col=1)
fig.add_trace(go.Scatter(x=hours, y=sza_day, name="solar zenith angle", line=dict(color=COLORS[3])), row=2, col=1)
fig.add_hline(y=90, line=dict(color="grey", dash="dot"), row=2, col=1)
fig.update_yaxes(title=f"{REACTIONS[0]} (s⁻¹)", row=1, col=1)
fig.update_yaxes(title="SZA (°)", row=2, col=1)
fig.update_xaxes(title="Hours since START_UTC", row=2, col=1)
plots.style(fig, f"{REACTIONS[0]} over one day at {LATITUDE:.1f}° latitude, {LONGITUDE:.1f}° longitude", height=520)

## 2. From a KPP mechanism to ODEs

The gas-phase chemistry is the MCM v3.3.1 limonene subset that Shantanu Jathar supplied as a
KPP equation file (`limonene_mcm.eqn`; KPP, the Kinetic PreProcessor, is a standard tool and file
format for chemical mechanisms). The model reads the file unchanged. Each reaction is
one line:

`<label> REACTANTS = PRODUCTS : rate expression ;`

The rate expression is Fortran: a number, an Arrhenius term in `TEMP`, a named MCM coefficient
such as `KMT03` (a pressure-dependent Troe expression), a photolysis frequency `J(J_NO2)`, or a
coefficient times `RO2`.

> <font color="#A47F52"><b>TRY THIS</b></font> &nbsp; Add `"<14815>"` (limonene + NO₃) to `SHOW`.

In [ ]:
#@title Choose the reaction lines to show
from tomas_jax.gaschem.config import LIMONENE_EQN
from tomas_jax.gaschem.kpp_parser import parse_eqn

lines = LIMONENE_EQN.read_text().splitlines()
# reaction labels to print (Try this: add "<14815>", limonene + NO3)
SHOW = ("<7>", "<12>", "<39>", "<13987>", "<13994>", "<14325>")  #@param {type:"raw"}
print("\n".join(lines[:3]), "\n...")
print("\n".join(line for line in lines if line.startswith(SHOW)))

raw = parse_eqn(LIMONENE_EQN)
print(f"\n{len(raw.species)} species, {len(raw.reactions)} reactions "
      f"({sum(r.photolysis for r in raw.reactions)} photolysis), {len(raw.ro2)} peroxy radicals in the RO2 sum")

The tutorial makes one edit, in code rather than in the file (ASSUMPTIONS A1): reaction `<34>`,
a fixed chamber sink for N₂O₅ (4×10⁻⁴ s⁻¹), gets its rate from the particles every interval
(Section 4), and its product is renamed `NA_HET` so the model can hand it to the particles as
nitrate. `<33>` (HNO₃ → NA) stays a gas-phase sink.

In [ ]:
from tomas_jax.box.coupling import N2O5_HET_LABEL, tutorial_mechanism

mech_def = tutorial_mechanism()            # the parsed file with the <34> edit
for parsed in (raw, mech_def):
    print(next(r for r in parsed.reactions if r.label == N2O5_HET_LABEL))

### Rate constants

Like KPP, the model evaluates every rate constant once per coupling interval and holds it fixed
while the solver integrates that interval. `RateModel` translates all 2135 expressions to Python
and, for a given temperature, pressure and RH, splits each into

$$k_j = k_j^\text{thermal}\,[\mathrm{RO_2}]^{\,s_j} + \sum_n c_{jn} J_n, \qquad s_j = 1 \text{ if the expression contains RO2, else } 0$$

so the update each interval is a multiply-add. Units: s⁻¹ for one reactant, cm³ molec⁻¹ s⁻¹
for two.

> <font color="#A47F52"><b>TRY THIS</b></font> &nbsp; Set `LABEL` to `"7"` (NO + O₃), `"12"` (NO₂ + NO₃ → N₂O₅, Troe fall-off) or `"13994"` (uses RO₂).

In [ ]:
#@title Choose a reaction label
from tomas_jax.gaschem.environment import air_environment
from tomas_jax.gaschem.mechanism import RateModel

level2 = at.scenario_from("level2_limonene_soa")
env2 = level2.environment
rate_model = RateModel(mech_def, external=(N2O5_HET_LABEL,))   # <34>: k comes from the aerosol
air = air_environment(env2.temp_k, env2.pres_pa, env2.rh)      # TEMP [K]; M, O2, N2, H2O [molec/cm3]
k_thermal, ro2_dependent, j_coef = rate_model.at_env(air)
print(f"M = {air['M']:.3e} molec/cm3. {ro2_dependent.sum()} reactions scale with RO2, "
      f"{(j_coef != 0).any(axis=1).sum()} with a J.\n")

# MCM reaction label (Try this: "7" (NO + O3), "12" (NO2 + NO3 -> N2O5, Troe fall-off), "13994" (uses RO2))
LABEL = "14325"  #@param {type:"string"}
i = rate_model.labels.index(LABEL)
rxn = mech_def.reactions[i]
print(f"<{LABEL}> {' + '.join(rxn.reactants)} -> {' + '.join(p for p, _ in rxn.products)}")
print(f"   expression : {rxn.rate_expr}")
print(f"   k_thermal  : {k_thermal[i]:.3e} at {env2.temp_k} K" + ("  (multiplied by RO2)" if ro2_dependent[i] else ""))
print(f"   J part     : {dict((rate_model.j_names[n], j_coef[i, n]) for n in np.flatnonzero(j_coef[i]))}")

temps = np.arange(260.0, 321.0, 5.0)
k_of_t = [rate_model.at_env(air_environment(t, env2.pres_pa, env2.rh))[0][i] for t in temps]
fig = go.Figure(go.Scatter(x=temps, y=k_of_t, mode="lines+markers", line=dict(color=COLORS[0])))
fig.update_xaxes(title="Temperature (K)")
unit = "s⁻¹" if len(rxn.reactants) == 1 else "cm³ molec⁻¹ s⁻¹"
fig.update_yaxes(title=f"k_thermal ({unit})", type="log", exponentformat="power")
plots.style(fig, f"<{LABEL}> rate constant vs temperature", height=340)

### The RO₂ sum

Peroxy radicals (RO₂) react with each other. Writing every RO₂ + RO₂′ pair would add about ten
thousand reactions, so MCM lets each peroxy radical react with the pool of all of them at
once: a pseudo-first-order rate $k_i\,[\mathrm{RO_2}]$ with
$[\mathrm{RO_2}] = \sum_m [\mathrm{RO_2}]_m$ over the 146 radicals listed in the file's
`#INLINE` block. The model takes $[\mathrm{RO_2}]$ at the start of each coupling interval and
holds it fixed for that interval, as KPP does (ASSUMPTIONS A12).

In [ ]:
print(f"{len(mech_def.ro2)} RO2 members, e.g. {', '.join(mech_def.ro2[:8])}, ...")
print("First reactions that use RO2:", ", ".join([f"<{lab}>" for lab, dep in zip(rate_model.labels, ro2_dependent) if dep][:8]))

### The ODEs

No reaction has more than two reactants, so every reaction rate is $r_j = k_j\,y_a\,y_b$
(molec cm⁻³ s⁻¹), and each species changes by production minus loss:

$$\frac{dy_i}{dt} = \sum_j \nu^\text{prod}_{ij}\, r_j \;-\; \sum_j \nu^\text{loss}_{ij}\, r_j$$

`Mechanism.build` turns the reaction list into index arrays, and `Mechanism.rhs(y, k)` computes
all 726 derivatives (the 725 species of the file plus the NA_HET counter) with two vectorised
sums in JAX. Below: the Level 2 starting gases under the
Level 2 constant light.

> <font color="#A47F52"><b>TRY THIS</b></font> &nbsp; Set `GASES_PPB = level2.gases_ppb | {"NO": 0.0, "NO2": 0.0}` (no NOx).

In [ ]:
from tomas_jax.gaschem.mechanism import EnvRates, Mechanism

mech = Mechanism.build(mech_def)
species = mech_def.species
index = {name: n for n, name in enumerate(species)}
GASES_PPB = level2.gases_ppb                    # Try this: level2.gases_ppb | {"NO": 0.0, "NO2": 0.0} (no NOx)
y0 = np.zeros(mech.n_species)
for name, ppb in GASES_PPB.items():
    y0[index[name]] = ppb * 1e-9 * air["M"]     # ppb -> molec/cm3

light2 = JProvider(rate_model.j_names, PhotolysisSettings(mode="constant", jno2=level2.photolysis.jno2_per_s,
                                                          constant_sza_deg=level2.photolysis.constant_sza_deg))
rates2 = EnvRates(jnp.asarray(k_thermal), jnp.asarray(ro2_dependent), jnp.asarray(j_coef))
k = rates2.k(mech.ro2(jnp.asarray(y0)), jnp.asarray(light2.at(0.0)))   # the frozen k for one interval
dydt = mech.rhs(jnp.asarray(y0), k)
print(GASES_PPB)
for name in ("LIMONENE", "O3", "NO", "NO2"):
    print(f"d[{name}]/dt at t = 0: {float(dydt[index[name]]) / air['M'] * 1e9 * 3600:+8.2f} ppb/h")

### Solving one coupling interval, and why it is stiff

`integrate_interval` is the call the driver makes every interval: it integrates the ODEs over
Δt with $k$ held fixed.

In [ ]:
from tomas_jax.gaschem.solver import integrate_interval

DT = level2.numerics.coupling_dt_s        # 300 s, the Level 2 coupling interval
t_wall = time.perf_counter()
y1, n_steps = integrate_interval(mech, jnp.asarray(y0), k, DT)
print(f"{DT:.0f} s of chemistry in {int(n_steps)} solver steps "
      f"({time.perf_counter() - t_wall:.1f} s wall time, including compilation the first time)")
for name in ("LIMONENE", "O3", "NO", "NO2"):
    print(f"  {name:9s} {y0[index[name]] / air['M'] * 1e9:7.3f} -> {float(y1[index[name]]) / air['M'] * 1e9:7.3f} ppb")
print(f"  OH        {y0[index['OH']]:.2e} -> {float(y1[index['OH']]):.2e} molec/cm3")

Look at O₃: at t = 0 it was disappearing at tens of ppb per hour, yet after 300 s it has gone
*up*. At t = 0 there are no O atoms, so only the losses act (O₃ photolysis, NO + O₃,
limonene + O₃). Within microseconds NO₂ + hν → NO + O and O + O₂ (+M) → O₃ catch up. Processes
that take microseconds next to ones that take hours: that is stiffness, and the next cells
measure it.

Linearise the ODEs around the current state: $d\,\delta\mathbf{y}/dt \approx \mathbf{J}\,\delta\mathbf{y}$,
with the Jacobian $J_{ik} = \partial(dy_i/dt)/\partial y_k$ (JAX computes it by automatic
differentiation). Each eigenvalue $\lambda$ of $\mathbf{J}$ is a mode that relaxes on the
timescale $1/|\mathrm{Re}\,\lambda|$. The diagonal gives a simpler number per species: its
lifetime against its own losses, $\tau_i = -1/J_{ii}$.

> <font color="#8B8D8E"><b>KEY IDEA</b></font> &nbsp; <mark>A system is **stiff** when these timescales span many orders of magnitude.</mark> The fastest mode sets the step size an explicit method needs to stay stable, even though we only care how the slow species change over minutes to hours.

In [ ]:
jac = np.asarray(jax.jacfwd(mech.rhs)(y1, k))           # 726 x 726
decay = -np.linalg.eigvals(jac).real
decay = decay[decay > 1e-10]                             # drop (near-)conserved modes
print(f"Eigenvalue timescales: {1 / decay.max():.1e} s to {1 / decay.min():.1e} s "
      f"(stiffness ratio {decay.max() / decay.min():.0e})")

loss = -np.diag(jac)
order = [n for n in np.argsort(loss)[::-1] if loss[n] > 0]
tau = 1.0 / loss[order]
fig = go.Figure(go.Scatter(x=np.arange(tau.size), y=tau, mode="markers", name="species",
                           marker=dict(size=4, color=COLORS[0]), text=[species[n] for n in order],
                           hovertemplate="%{text}: τ = %{y:.2e} s<extra></extra>"))
for n, (label, seconds) in enumerate([("1 µs", 1e-6), ("1 s", 1.0), (f"coupling interval ({DT:.0f} s)", DT),
                                      ("1 day", 86400.0)]):
    fig.add_trace(go.Scatter(x=[0, tau.size], y=[seconds] * 2, mode="lines", name=label,
                             line=dict(color=COLORS[n + 1], dash="dot")))
fig.update_yaxes(type="log", title="τ = −1/J_ii (s)", exponentformat="power")
fig.update_xaxes(title="Species, fastest first (hover for the name)")
plots.style(fig, f"Chemical lifetimes after {DT:.0f} s of Level 2 chemistry", height=460)

> <font color="#858F6A"><b>WHAT TO NOTICE</b></font> &nbsp; The flat steps are families of species (hover to see them): about a hundred alkoxy radicals (RO) that decompose within a microsecond, Criegee intermediates and acyl peroxy radicals (RCO₃) at 0.1–2 s, peroxy radicals (RO₂) at about 20 s (mostly RO₂ + NO), PAN-type species at about half an hour, and closed-shell products from hours to weeks.

An explicit method (forward Euler, classic Runge–Kutta) is only stable for
$\Delta t \lesssim 2/|\lambda_\text{max}|$, here a few nanoseconds: one 300 s interval would
need about 10¹¹ steps. <mark>An implicit method solves a linear system with the Jacobian at every step, which keeps it stable at any step size, so accuracy alone sets the step.</mark> The model uses
diffrax's **Kvaerno3** (a third-order implicit Runge–Kutta method) with adaptive steps
(relative tolerance 10⁻³, absolute tolerance 10⁴ molec cm⁻³ ≈ 0.4 ppq, parts per quadrillion).

The cell below calls diffrax the way `integrate_interval` does, but saves every step so we can
look at the step sizes. It also tries the explicit Tsit5 solver with the same budget of 2000
steps.

> <font color="#A47F52"><b>TRY THIS</b></font> &nbsp; Set `RTOL = 1e-5` or `ATOL = 1.0` (molec/cm³), or add `"Kvaerno5 (implicit, 5th order)": dfx.Kvaerno5()` to `SOLVERS`.

In [ ]:
from tomas_jax.gaschem.config import SOLVER_ATOL, SOLVER_MAX_STEPS, SOLVER_RTOL


@eqx.filter_jit
def solve_interval(y0, k, dt, rtol, atol, solver):
    """Same as integrate_interval, but saves every step (unused slots are inf)."""
    term = dfx.ODETerm(lambda t, y, k: mech.rhs(y, k))
    return dfx.diffeqsolve(term, solver, t0=0.0, t1=dt, dt0=None, y0=y0, args=k,
                           stepsize_controller=dfx.PIDController(rtol=rtol, atol=atol),
                           saveat=dfx.SaveAt(steps=True), max_steps=SOLVER_MAX_STEPS, throw=False)


RTOL, ATOL = SOLVER_RTOL, SOLVER_ATOL       # Try this: RTOL = 1e-5, or ATOL = 1.0 (molec/cm3)
SOLVERS = {"Kvaerno3 (implicit, the model's choice)": dfx.Kvaerno3(),
           "Tsit5 (explicit)": dfx.Tsit5()}   # Try this: add "Kvaerno5 (implicit, 5th order)": dfx.Kvaerno5()
step_times = {}
for label, solver in SOLVERS.items():
    t_wall = time.perf_counter()
    sol = solve_interval(jnp.asarray(y0), k, DT, jnp.float64(RTOL), jnp.float64(ATOL), solver)
    ts = np.asarray(sol.ts)
    step_times[label] = ts[np.isfinite(ts)]
    status = "finished" if sol.result == dfx.RESULTS.successful else "ran out of steps"
    print(f"{label}: {status} at t = {step_times[label][-1]:.3g} s of {DT:.0f} s; "
          f"{int(sol.stats['num_accepted_steps'])} accepted + {int(sol.stats['num_rejected_steps'])} rejected steps; "
          f"{time.perf_counter() - t_wall:.1f} s wall time")

fig = go.Figure()
for n, (label, ts) in enumerate(step_times.items()):
    fig.add_trace(go.Scatter(x=ts, y=np.diff(ts, prepend=0.0), mode="lines+markers", name=label,
                             marker=dict(size=4), line=dict(color=COLORS[n])))
fig.update_xaxes(type="log", title="Time within the interval (s)", exponentformat="power")
fig.update_yaxes(type="log", title="Step size (s)", exponentformat="power")
plots.style(fig, "Accepted step sizes over one coupling interval", height=420)

> <font color="#858F6A"><b>WHAT TO NOTICE</b></font> &nbsp; With the model's tolerances, Kvaerno3 starts with steps of about 0.01 s while the radicals spin up from zero, then lengthens them to about a minute and needs only a couple of dozen steps for the whole interval. Stability holds Tsit5 to steps of a few nanoseconds, so it uses up its 2000 steps within the first few microseconds.

## 3. Absorptive partitioning: Pankow and c*

> <font color="#8B8D8E"><b>KEY IDEA</b></font> &nbsp; Each condensable organic $i$ has a saturation mass concentration $c^*_i$ (µg m⁻³). In absorptive partitioning theory ([Pankow, 1994](https://doi.org/10.1016/1352-2310%2894%2990093-0); [Donahue et al., 2006](https://doi.org/10.1021/es052297c)) its particle-phase fraction at equilibrium is
>
> $$F_i = \left(1 + \frac{c^*_i}{C_{OA}}\right)^{-1}, \qquad C_{OA} = C_{nv} + \sum_i C_{\text{tot},i}\,F_i$$
>
> $C_{OA}$ is the absorbing organic aerosol mass, including non-volatile OA $C_{nv}$ (in the model, the organic part of the seed). $C_{OA}$ depends on the $F_i$, so the second equation is solved by iteration.

$c^*$ depends on temperature through Clausius–Clapeyron, with
$\Delta H_\text{vap} = 131 - 11\log_{10} c^*$ kJ mol⁻¹ ([Epstein et al., 2010](https://doi.org/10.1021/es902497z)).

A species with $c^* = C_{OA}$ is half in the particles. With ambient $C_{OA}$ of about
1–10 µg m⁻³, species with $c^*$ between about 0.1 and 100 µg m⁻³ are the ones whose split
responds to the amount of OA.

<details>
<summary><b>Where the model's c* values come from</b></summary>

In Levels 2 and 3 each limonene product gets its $c^*$ from a per-species table supplied by
Shantanu Jathar. He estimated the pure-component vapour pressure of every species at 298.15 K
with the group-contribution method of [Nannoolal et al. (2008)](https://doi.org/10.1016/j.fluid.2008.04.020) in the PyCHAM
box model ([O'Meara et al., 2021](https://doi.org/10.5194/gmd-14-675-2021)) and converted it to

$$c^*_i = \frac{10^6\, M_i\, p^\circ_i}{R\,T}$$

with $M_i$ the molar mass (g mol⁻¹), $p^\circ_i$ the vapour pressure (Pa), $R$ = 8.314 J mol⁻¹ K⁻¹
and $T$ = 298.15 K; the factor 10⁶ turns g m⁻³ into µg m⁻³. Of the 458 closed-shell organics in
the table, 228 have a $c^*$ of 10^4.25 µg m⁻³ or less (the top of the highest class) and can
condense. The four the table leaves out (limonene, CO, CH₄ and the OH tracer D9) get the
composition-only estimate of [Li et al. (2016)](https://doi.org/10.5194/acp-16-3327-2016), and
none of them condenses. With these values about 90 % of the Level 2 SOA is hydroperoxides and
5 % contains nitrogen.

> <font color="#8B8D8E"><b>CAVEAT</b></font> &nbsp; Group-contribution methods can disagree by a decade or more for molecules with several functional groups, so $c^*$ is still the most uncertain input of Levels 2 and 3 (ASSUMPTIONS A8).

</details>

> <font color="#A47F52"><b>TRY THIS</b></font> &nbsp; Change the classes in `CSTAR` (up to 15).

In [ ]:
CSTAR = np.array([0.01, 0.1, 1.0, 10.0, 100.0, 1000.0])   # µg/m3; Try this: other classes (up to 15)
c_oa = np.logspace(-1, 3, 200)
fig = go.Figure()
for n, cstar in enumerate(CSTAR):
    fig.add_trace(go.Scatter(x=c_oa, y=1 / (1 + cstar / c_oa), name=f"c* = {cstar:g}",
                             line=dict(color=COLORS[n % len(COLORS)])))
fig.add_vrect(x0=1, x1=10, fillcolor="grey", opacity=0.12, line_width=0)
fig.update_xaxes(type="log", title="C_OA (µg m⁻³); grey band: typical ambient range")
fig.update_yaxes(title="Particle fraction F", range=[0, 1.02])
plots.style(fig, "Equilibrium particle fraction F = 1 / (1 + c*/C_OA)", height=420)

### Does the model's partitioning code reach this equilibrium?

The model never sets $F$ directly. In every 60 s micro-step, `soa_partition_step` (a TOMAS
adapter around [vbs-jax](https://github.com/reflective-org/vbs-jax)) moves mass between the gas
and each size bin at a rate set by the condensation sink and the driving force: gas
concentration minus the saturation concentration over that particle, which includes Raoult's law
and the Kelvin effect ([Zaveri et al., 2014](https://doi.org/10.5194/acp-14-5153-2014)). Given time, it should relax to the Pankow solution.
Below we call it directly with 600 s steps; it splits them into sub-steps as needed.

Test: one volatility class per `CSTAR` value above, each with a fixed total (0.5 to 16 µg m⁻³), all
starting in the gas phase, on a seed
that carries $C_{nv}$ = 2 µg m⁻³ of non-volatile OA, with only partitioning switched on for 4 h.
The seed particles are 1.5 µm so the Kelvin effect, which the Pankow equation ignores, stays
below 1 %. Under a second.

> <font color="#A47F52"><b>TRY THIS</b></font> &nbsp; Set `C_TOT = C_TOT * 10`, `C_NV` to 0.2 or 20, `SEED = SeedMode(2000.0, 80.0, 1.6)` (Kelvin matters) or `TEMPERATURE_K = 278.0` (c\* drops 20–80×).

In [ ]:
#@title Set up the partitioning test and run it
from tomas_jax.box.coupling import kg_cell_to_ugm3, mass_grid, seed_distribution, ugm3_to_kg_cell
from tomas_jax.box.scenario import SeedMode
from tomas_jax.core.config import N_GAS_SPECIES, SRTORG_NUC, SRTSO4, SRTSOA1
from tomas_jax.gaschem.config import DHVAP_A_KJ, DHVAP_B_KJ
from tomas_jax.physics.soa_partitioning import soa_partition_step
from vbs_jax.config import calc_cstar_T

C_TOT = np.geomspace(0.5, 16.0, CSTAR.size)        # gas + particle per class, µg/m3 (one per CSTAR); Try this: C_TOT * 10
# non-volatile absorbing OA, µg/m3; Try this: 0.2 or 20
C_NV = 2.0  #@param {type:"raw"}
SEED = SeedMode(number_cm3=20.0, gmd_nm=1500.0, gsd=1.2)   # Try this: SeedMode(2000.0, 80.0, 1.6) (Kelvin matters)
# K (Try this: 278.0; c* drops 20-80x)
TEMPERATURE_K = 298.0  #@param {type:"raw"}
HOURS, STEP_S = 4, 600.0

dhvap = DHVAP_A_KJ - DHVAP_B_KJ * np.log10(CSTAR)  # kJ/mol, as the model assigns it
xk = mass_grid(40)                                  # TOMAS bin edges (kg per particle)
nk, mk = seed_distribution([SEED], 0.0, xk, BOXVOL)  # sulfate seed: number and mass per bin
mk[:, SRTORG_NUC] = ugm3_to_kg_cell(C_NV, BOXVOL) * mk[:, SRTSO4] / mk[:, SRTSO4].sum()
gc = np.zeros(N_GAS_SPECIES)
gc[SRTSOA1:SRTSOA1 + CSTAR.size] = ugm3_to_kg_cell(C_TOT, BOXVOL)   # every class starts as vapour

step = jax.jit(soa_partition_step)
Nk, Mk, Gc = jnp.asarray(nk), jnp.asarray(mk), jnp.asarray(gc)
soa_slots = slice(SRTSOA1, SRTSOA1 + CSTAR.size)
times, particle = [0.0], [np.zeros(CSTAR.size)]
for n in range(int(HOURS * 3600 / STEP_S)):
    Nk, Mk, Gc, _ = step(Nk, Mk, Gc, jnp.asarray(xk), TEMPERATURE_K, env2.pres_pa, BOXVOL, STEP_S,
                         jnp.asarray(CSTAR), jnp.asarray(dhvap), jnp.full(CSTAR.size, 200.0),  # MW 200 g/mol
                         1.0)                                                                 # accommodation
    times.append((n + 1) * STEP_S / 3600)
    particle.append(kg_cell_to_ugm3(np.asarray(Mk[:, soa_slots]).sum(axis=0), BOXVOL))
particle = np.array(particle)                       # [time, class], µg/m3

cstar_t = np.asarray(calc_cstar_T(CSTAR, dhvap, TEMPERATURE_K))   # c* at this temperature
c_oa_eq = C_NV + C_TOT.sum()
for _ in range(200):                                # fixed-point iteration of the C_OA equation
    c_oa_eq = C_NV + np.sum(C_TOT / (1 + cstar_t / c_oa_eq))
pankow = C_TOT / (1 + cstar_t / c_oa_eq)
print(f"C_OA: Pankow {c_oa_eq:.3f}, model after {HOURS} h {C_NV + particle[-1].sum():.3f} µg/m3")
for cstar, eq, mod in zip(CSTAR, pankow, particle[-1]):
    print(f"  c* = {cstar:7g}: Pankow {eq:7.3f}   model {mod:7.3f} µg/m3   ({100 * (mod / eq - 1):+.1f} %)")

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Particle mass per class at the end", "Approach to equilibrium"))
cstar_labels = [f"{c:g}" for c in CSTAR]
fig.add_trace(go.Bar(x=cstar_labels, y=pankow, name="Pankow equilibrium", marker_color=config.INITIAL_COLOR), row=1, col=1)
fig.add_trace(go.Bar(x=cstar_labels, y=particle[-1], name=f"model after {HOURS} h", marker_color=config.PARTICLE_COLOR),
              row=1, col=1)
fig.add_trace(go.Scatter(x=times, y=particle.sum(axis=1), name="model SOA", line=dict(color=config.PARTICLE_COLOR)),
              row=1, col=2)
fig.add_trace(go.Scatter(x=[0, HOURS], y=[pankow.sum()] * 2, name="Pankow SOA", mode="lines",
                         line=dict(color=config.INITIAL_COLOR, dash="dash")), row=1, col=2)
fig.update_xaxes(title="c* at 298 K (µg m⁻³)", type="category", row=1, col=1)
fig.update_xaxes(title="Time (h)", row=1, col=2)
fig.update_yaxes(title="Particle phase (µg m⁻³)", row=1, col=1)
fig.update_yaxes(title="SOA (µg m⁻³)", row=1, col=2)
plots.style(fig, "Model partitioning vs the Pankow equilibrium", height=430, legend_right=True)

> <font color="#858F6A"><b>WHAT TO NOTICE</b></font> &nbsp; Low-$c^*$ classes condense almost completely, $c^*$ = 1000 mostly stays in the gas, and the classes near $C_{OA}$ split. The model ends within 1 % of Pankow; the small shortfall is the Kelvin effect. With 80 nm seeds (try it) the Kelvin term raises the vapour pressure by about 9 % over an 80 nm particle, and more over the smaller particles of the mode, so the more volatile classes end 3–4 % below Pankow.

## 4. N₂O₅ uptake: from γ to a rate constant

At night NO₃ and N₂O₅ build up (NO₂ + O₃ → NO₃; NO₃ + NO₂ ⇌ N₂O₅). N₂O₅ that hits a wet
particle can react to 2 HNO₃, which stays in the particle as nitrate.

> <font color="#8B8D8E"><b>KEY IDEA</b></font> &nbsp; The uptake coefficient γ is the probability that a collision leads to reaction. From kinetic gas theory, molecules hit a unit area at a rate $\bar c/4$ per unit concentration, so the first-order loss rate is
>
> $$k_\text{het} = \frac{\gamma\,\bar c\,A}{4}, \qquad \bar c = \sqrt{\frac{8RT}{\pi M}}$$
>
> with $A$ the particle surface area per volume of air.

That holds when gas diffusion to the
particle is not the bottleneck: in the free-molecular limit, for particles small compared with the
mean free path λ ≈ 100 nm (Knudsen number Kn = 2λ/D_p ≫ 1), or more generally when γ ≪ Kn, i.e.
when the surface reaction is slow compared with diffusion. For larger particles and larger γ,
diffusion to the surface also limits the rate, so the model uses the TOMAS condensation sink with
γ in place of the accommodation coefficient:

$$k_\text{het} = 2\pi D_g \sum_k D_{p,k}\, N_k\, \beta(\mathrm{Kn}_k, \gamma)$$

with the Fuchs–Sutugin factor β, which reduces to the formula above when Kn ≫ 1 or γ ≪ Kn. Diameters are
wet, so particle water adds surface. In the tutorial mechanism this $k_\text{het}$ replaces the
chamber's fixed N₂O₅ sink `<34>` (4×10⁻⁴ s⁻¹) in every coupling interval (ASSUMPTIONS A1, A5).

> <font color="#A47F52"><b>TRY THIS</b></font> &nbsp; Add `1.0` to `GAMMAS`, set `NUMBER_CM3 = 30000`, or set `RH = 0.9` (Level 3 uses 0.6).

In [ ]:
#@title Choose the uptake coefficients and sweep the seed size
from tomas_jax.core.config import MW_N2O5, PI, R_GAS
from tomas_jax.physics.condensation_sink import calc_wet_diameters
from tomas_jax.physics.heterogeneous import n2o5_uptake
from tomas_jax.physics.water_equilibrium import equilibrate_water

# N₂O₅ uptake coefficients γ (Try this: add 1.0)
GAMMAS = [0.002, 0.02, 0.1]  #@param {type:"raw"}
NUMBER_CM3, GSD = 3000.0, 1.6             # the Level 3 seed's number and width; Try this: NUMBER_CM3 = 30000
GMD_NM = np.geomspace(10, 1000, 15)       # seed diameters to sweep
RH = level3.environment.rh                # 0.6 in Level 3; Try this: 0.9
env3 = level3.environment
uptake = jax.jit(n2o5_uptake)
c_bar = np.sqrt(8 * R_GAS * env3.temp_k / (PI * MW_N2O5 * 1e-3))   # m/s


def area_and_k_het(modes, organic_fraction, gamma):
    """Wet surface area (m2 per m3 of air) and the model's k_het (1/s) for a seed built as run() does."""
    nk, mk = seed_distribution(modes, organic_fraction, xk, BOXVOL)
    mk = equilibrate_water(jnp.asarray(mk), RH, env3.temp_k, AerosolStepConfig().water_scheme)
    dp, _ = calc_wet_diameters(jnp.asarray(nk), mk, jnp.asarray(xk))        # wet diameter per bin (m)
    area = float(jnp.sum(PI * dp**2 * nk)) / (BOXVOL * 1e-6)
    k_het, _ = uptake(jnp.asarray(nk), mk, jnp.asarray(xk), env3.temp_k, env3.pres_pa, BOXVOL, gamma)
    return area, float(k_het)


sweep = np.array([(gmd, gamma, *area_and_k_het([SeedMode(NUMBER_CM3, gmd, GSD)], 0.3, gamma))
                  for gmd in GMD_NM for gamma in GAMMAS])          # columns: gmd, gamma, area, k_het
area3, k3 = area_and_k_het(level3.seed.modes, level3.seed.organic_mass_fraction, level3.gamma_n2o5)
print(f"c_bar(N2O5) = {c_bar:.0f} m/s at {env3.temp_k} K")
print(f"Level 3 seed at RH {RH:g}: A = {area3 * 1e6:.0f} µm²/cm³; with γ = {level3.gamma_n2o5:g}, "
      f"k_het = {k3:.2e} 1/s, an N2O5 lifetime against uptake of {1 / k3 / 3600:.1f} h "
      f"(γ c̄ A/4 gives {level3.gamma_n2o5 * c_bar * area3 / 4:.2e} 1/s)")

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("k_het vs wet surface area", "Model / (γ c̄ A / 4)"))
for n, gamma in enumerate(GAMMAS):
    rows = sweep[sweep[:, 1] == gamma]
    gmd, area_um2_cm3, k_model = rows[:, 0], rows[:, 2] * 1e6, rows[:, 3]
    k_free = gamma * c_bar * rows[:, 2] / 4
    color = COLORS[n % len(COLORS)]
    fig.add_trace(go.Scatter(x=area_um2_cm3, y=k_free, name=f"γ = {gamma:g}: γ c̄ A / 4", legendgroup=str(gamma),
                             mode="lines", line=dict(color=color, dash="dash")), row=1, col=1)
    fig.add_trace(go.Scatter(x=area_um2_cm3, y=k_model, name=f"γ = {gamma:g}: model", legendgroup=str(gamma),
                             mode="markers", marker=dict(color=color), customdata=gmd,
                             hovertemplate="GMD %{customdata:.0f} nm<br>A %{x:.3g} µm² cm⁻³<br>k %{y:.2e} s⁻¹"),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=gmd, y=k_model / k_free, name=f"γ = {gamma:g}", legendgroup=str(gamma),
                             showlegend=False, line=dict(color=color)), row=1, col=2)
span = [sweep[:, 2].min() * 1e6, sweep[:, 2].max() * 1e6]
fig.add_trace(go.Scatter(x=span, y=[4e-4, 4e-4], name="chamber sink <34> (replaced)", mode="lines",
                         line=dict(color="grey", dash="dot")), row=1, col=1)
fig.update_xaxes(type="log", title="A (µm² cm⁻³)", exponentformat="power", row=1, col=1)
fig.update_yaxes(type="log", title="k_het (s⁻¹)", exponentformat="power", row=1, col=1)
fig.update_xaxes(type="log", title="Seed GMD (nm), dry", row=1, col=2)
fig.update_yaxes(title="ratio", row=1, col=2)
plots.style(fig, f"N₂O₅ uptake for {NUMBER_CM3:g} particles cm⁻³ of growing size (RH {RH:g})", height=460,
      legend_right=True)

> <font color="#858F6A"><b>WHAT TO NOTICE</b></font> &nbsp; For small particles the model sits on the dashed $\gamma \bar c A/4$ lines. As particles grow past the mean free path, gas diffusion starts to limit uptake and the ratio drops below 1, most for large γ, because a fast surface reaction depletes N₂O₅ near the particle. In the Level 3 run, $k_\text{het}$ also rises over the day as sulfate, nitrate and their water add surface (Section 5 shows it).

## 5. How the pieces are coupled: operator splitting

Gas chemistry (concentrations in molec cm⁻³, stiff ODEs) and aerosol (TOMAS size bins, mass and
number per box) are advanced one after the other, not together. For each coupling interval
Δt_c (300 s in Levels 1–2, 900 s in Level 3) the driver does:

```text
 state at t
   │
   ├─ 1. rate constants for the interval, then held fixed:
   │       J at the interval midpoint (Section 1), RO2 at t (Section 2),
   │       k_het(N2O5) from the particles at t (Section 4)
   │
   ├─ 2. gas chemistry over Δt_c with the stiff solver (Section 2)
   │       two counter species record what was formed for the particles:
   │       SA = H2SO4 (from SO2 + OH), NA_HET = nitrate (from N2O5 uptake)
   │
   ├─ 3. gather: condensable organics ──► their c* classes (gas side)
   │
   ├─ 4. aerosol over Δt_c in 60 s micro-steps:
   │       sources (H2SO4, nitrate) → nucleation → coagulation →
   │       H2SO4 condensation → SOA partitioning (Section 3) → water uptake
   │
   ├─ 5. scatter: the change in each class's gas ──► back to the individual species;
   │       reset "hold constant" species (Level 3: NO2 at 1 ppb, a stand-in for NOx emissions)
   ▼
 state at t + Δt_c
```

<mark>Splitting is accurate when little changes within Δt_c</mark>; the error shrinks with Δt_c. Going from
900 s to 600 s changes the Level 3 nitrate at 24 h by 2.6 % and the HNO₃ by 3.1 %, more early in
the run and around sunset (ASSUMPTIONS A12). The SOA at 24 h changes by 10 %, because so little is
left (0.06 µg m⁻³) that small differences in how much evaporates on the second morning show up
large. Level 1 has no gas chemistry and skips steps 1–3 and 5.

First, the result of all this for the Level 3 preset, precomputed (no model run). The SOA stays
close to zero: the 2 ppb of limonene make at most 0.39 µg m⁻³, 2 hours in. After that, OH and
sunlight destroy the hydroperoxides in the gas phase. At equilibrium each species splits between
gas and particles as $F_i = (1 + c^*_i/C_{OA})^{-1}$ says (Section 3), so when its gas-phase part is
destroyed, molecules evaporate from the particles to restore that split; as $C_{OA}$ shrinks, $F_i$
falls too. 0.06 µg m⁻³ is left at 24 h.

In [ ]:
example = at.load_example("level3_day_night")
plots.time_series([example], ["J_NO2", "N2O5", "NIT_ugm3", "SOA_ugm3"])

### The loop, written out

The next two cells rebuild `tomas_jax.box.driver.run` from the pieces used above: first the
set-up, then the five steps for every interval. It is a stripped-down copy of the driver (no
output recording, no warnings), but it makes the same calls in the same order, so it should
reproduce the model. The Level 3 preset (24 h, 96 intervals) takes about 12 s on a laptop with
40 bins, a few seconds longer the first time while JAX compiles the aerosol step.

In [ ]:
from tomas_jax.box.aerosol_step import AEROSOL_PROCESSES, AerosolForcing, AerosolState, SOAParams, aerosol_outer_step
from tomas_jax.box.coupling import NA_HET, ClassBridge, cm3_to_kg_cell
from tomas_jax.core.config import MW_H2SO4, MW_NO3, SRTNIT
from tomas_jax.gaschem.config import AVOGADRO, CSTAR_TABLE_SOURCES, MECHANISM_DIR
from tomas_jax.gaschem.volatility import assign_classes, load_cstar_table

s = at.scenario_from("level3_day_night")    # Try this: at.scenario_from("level3_day_night", gamma_n2o5=0.1)
cfg = AerosolStepConfig()                   # box volume 1e10 cm3, 60 s aerosol micro-steps
run_env, dt = s.environment, s.numerics.coupling_dt_s

# Gas side: rate-constant pieces at this T, P, RH; light for the site and date; initial gases
run_air = air_environment(run_env.temp_k, run_env.pres_pa, run_env.rh)
rates = EnvRates(*(jnp.asarray(a) for a in rate_model.at_env(run_air)))
run_light = JProvider(rate_model.j_names, PhotolysisSettings(
    mode=s.photolysis.mode, latitude=s.photolysis.latitude_deg, longitude=s.photolysis.longitude_deg,
    start_utc=s.start_utc, jno2=s.photolysis.jno2_per_s, constant_sza_deg=s.photolysis.constant_sza_deg))
y = np.zeros(mech.n_species)
for name, ppb in s.gases_ppb.items():
    y[index[name]] = ppb * 1e-9 * run_air["M"]
held = [index[name] for name in s.hold_constant]
held_values = y[held].copy()

# Aerosol side: c* classes of the mechanism's products, seed with water.
# s.cstar_table names the c* table: the presets use Shantanu Jathar's (Section 3); None gives
# every species the Li et al. (2016) composition estimate.
if s.cstar_table is None:
    classes = assign_classes(mech_def)
else:
    folder = MECHANISM_DIR if (MECHANISM_DIR / s.cstar_table).exists() else config.CSTAR_TABLE_DIR
    log10_cstar = load_cstar_table(folder / s.cstar_table)          # {species: log10 c*}
    classes = assign_classes(mech_def, log10_cstar, CSTAR_TABLE_SOURCES.get(s.cstar_table, f"table {s.cstar_table}"))
n_cls = classes.class_cstar.size
soa = SOAParams(jnp.asarray(classes.class_cstar), jnp.asarray(classes.class_dhvap),
                jnp.asarray(classes.class_mw), jnp.float64(1.0))
bridge = ClassBridge(classes, cfg.boxvol_cm3)      # moves organics between species and classes
run_xk = mass_grid(s.numerics.nbins)
run_nk, run_mk = seed_distribution(s.seed.modes, s.seed.organic_mass_fraction, run_xk, cfg.boxvol_cm3)
run_mk = equilibrate_water(jnp.asarray(run_mk), run_env.rh, run_env.temp_k, cfg.water_scheme)
aerosol = AerosolState(jnp.asarray(run_nk), run_mk, jnp.zeros(N_GAS_SPECIES), jnp.asarray(run_xk),
                       jnp.float64(run_env.temp_k), jnp.float64(run_env.pres_pa), jnp.float64(run_env.rh),
                       jnp.float64(cfg.boxvol_cm3))
processes = tuple(p for p in AEROSOL_PROCESSES if getattr(s.processes, p))
run_gamma = s.gamma_n2o5 if s.processes.n2o5_uptake else 0.0
i_het = rate_model.labels.index(N2O5_HET_LABEL)


def ugm3(molec_cm3, mw):
    """molec/cm3 -> µg/m3."""
    return molec_cm3 * mw / AVOGADRO * 1e12


print(f"{s.name}: {s.duration_h:g} h in {round(s.duration_h * 3600 / dt)} intervals of {dt:.0f} s; "
      f"{n_cls} c* classes ({len(classes.species)} condensable species); aerosol processes: {', '.join(processes)}")
print(f"c* source: {classes.source}")

In [ ]:
log = []
t_wall = time.monotonic()
for n_int in range(round(s.duration_h * 3600 / dt)):
    if time.monotonic() - t_wall > config.RUN_TIMEOUT_S:
        raise TimeoutError(f"the loop took longer than {config.RUN_TIMEOUT_S:.0f} s")
    t0 = n_int * dt
    # 1. rate constants for this interval
    j_now = run_light.at(t0 + 0.5 * dt)
    k_int = rates.k(mech.ro2(jnp.asarray(y)), jnp.asarray(j_now))
    k_het, weights = n2o5_uptake(aerosol.Nk, aerosol.Mk, aerosol.xk, aerosol.temp, aerosol.pres,
                                 aerosol.boxvol, jnp.float64(run_gamma))
    k_int = k_int.at[i_het].set(k_het)
    # 2. gas chemistry; read and reset the two counters
    y_new, n_steps = integrate_interval(mech, jnp.asarray(y), k_int, dt)
    y = np.array(y_new)
    sa, na_het = y[index["SA"]], y[index[NA_HET]]
    y[index["SA"]] = y[index[NA_HET]] = 0.0
    # 3. gather condensable organics into the c* classes
    gas_before = bridge.gas_mass(y)
    aerosol = aerosol._replace(Gc=aerosol.Gc.at[SRTSOA1:SRTSOA1 + n_cls].set(bridge.class_sum(gas_before)))
    # 4. aerosol; H2SO4 and nitrate enter as constant sources (kg per box per s) over the interval
    forcing = AerosolForcing(h2so4_rate=jnp.float64(cm3_to_kg_cell(sa, MW_H2SO4, cfg.boxvol_cm3) / dt),
                             org_rate=jnp.zeros(n_cls),
                             nitrate_rate=jnp.float64(cm3_to_kg_cell(na_het, MW_NO3, cfg.boxvol_cm3) / dt),
                             nitrate_weights=weights)
    aerosol, _ = aerosol_outer_step(aerosol, forcing, soa, dt, processes=processes, cfg=cfg)
    # 5. scatter the class gas change back to the species; reset held species
    mk_now = np.asarray(aerosol.Mk)
    y = bridge.scatter(y, gas_before, np.asarray(aerosol.Gc[SRTSOA1:SRTSOA1 + n_cls]),
                       mk_now[:, SRTSOA1:SRTSOA1 + n_cls].sum(axis=0))
    y[held] = held_values
    log.append({"t_mid_h": (t0 + 0.5 * dt) / 3600, "J_NO2": j_now[rate_model.j_names.index("J_NO2")],
                "k_het": float(k_het), "steps": int(n_steps),
                "h2so4": ugm3(sa, MW_H2SO4), "nitrate": ugm3(na_het, MW_NO3)})
print(f"{len(log)} intervals in {time.monotonic() - t_wall:.1f} s")

### Check against the model

For the unchanged preset at 40 bins we compare with the precomputed result (rounded to 6 digits).
If you changed `s` or `N_BINS`, the cell runs the model with `at.run(s)` instead (about another
15 s at 40 bins).

In [ ]:
reference = example if s == at.scenario_from(example) else at.run(s)   # the example has 40 bins
mk_end = np.asarray(aerosol.Mk)
check = {
    "particle nitrate (µg/m³)": (kg_cell_to_ugm3(mk_end[:, SRTNIT].sum(), cfg.boxvol_cm3),
                                 reference.totals["NIT_ugm3"][-1]),
    "SOA (µg/m³)": (kg_cell_to_ugm3(mk_end[:, SRTSOA1:SRTSOA1 + n_cls].sum(), cfg.boxvol_cm3),
                    reference.totals["SOA_ugm3"][-1]),
    "particle number (cm⁻³)": (float(aerosol.Nk.sum()) / cfg.boxvol_cm3, reference.totals["N_cm3"][-1]),
    "O₃ (ppb)": (y[index["O3"]] / run_air["M"] * 1e9, reference.gases_ppb["O3"][-1]),
    "HNO₃ (ppb)": (y[index["HNO3"]] / run_air["M"] * 1e9, reference.gases_ppb["HNO3"][-1]),
    "gas solver steps": (sum(e["steps"] for e in log), reference.diagnostics["gas_solver_steps"]),
}
print(f"{'at the end of the run':26s} {'our loop':>12s} {'model':>12s}")
for name, (ours, model) in check.items():
    print(f"{name:26s} {ours:12.6g} {model:12.6g}")

In [ ]:
t_mid = [e["t_mid_h"] for e in log]
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=("Step 1: J(NO₂) used for the interval",
                                    "Step 1: N₂O₅ uptake rate k_het from the particles",
                                    "Step 2: gas solver steps per interval",
                                    "Step 4: handed to the aerosol per interval"))
fig.add_trace(go.Scatter(x=t_mid, y=[e["J_NO2"] for e in log], name="J(NO₂)", line=dict(color=COLORS[3])), 1, 1)
fig.add_trace(go.Scatter(x=t_mid, y=[e["k_het"] for e in log], name="k_het", line=dict(color=COLORS[6])), 2, 1)
fig.add_trace(go.Bar(x=t_mid, y=[e["steps"] for e in log], name="solver steps", marker_color="#888888"), 3, 1)
layer_color = {key: color for key, _, color in config.COMPOSITION_LAYERS}
fig.add_trace(go.Scatter(x=t_mid, y=[e["h2so4"] for e in log], name="H₂SO₄",
                         line=dict(color=layer_color["SO4_ugm3"])), 4, 1)
fig.add_trace(go.Scatter(x=t_mid, y=[e["nitrate"] for e in log], name="nitrate",
                         line=dict(color=layer_color["NIT_ugm3"])), 4, 1)
for start, end in at.night_intervals(reference):
    fig.add_vrect(x0=start, x1=end, fillcolor=config.NIGHT_COLOR, line_width=0, row="all", col=1)
for row, title in enumerate(["J(NO₂)<br>(s⁻¹)", "k_het<br>(s⁻¹)", "steps",
                             f"µg m⁻³ per<br>{dt / 60:g} min"], start=1):
    fig.update_yaxes(title=title, row=row, col=1)
fig.update_xaxes(title=config.TIME_AXIS_TITLE, row=4, col=1)
fig.update_yaxes(exponentformat="power")
plots.style(fig, f"{s.name}: what happens in each coupling interval (shaded: night)", height=860, legend_right=True)

> <font color="#858F6A"><b>WHAT TO NOTICE</b></font> &nbsp;
>
> - **J** comes from the sun's position at the middle of each interval; it is zero at night.
> - **k_het** rises only about 1.2-fold in the first four hours (1.5×10⁻⁴ to 1.9×10⁻⁴ s⁻¹), as sulfate, its water and a little SOA add surface (after 4 hours the particles hold about 0.85 µg m⁻³ more sulfate, 0.5 more water and 0.3 of SOA). It rises again during the night as nitrate is added, and once more on the second morning with new sulfate. It ends at 3.1×10⁻⁴ s⁻¹, about a quarter below the chamber's fixed 4×10⁻⁴ s⁻¹.
> - **Solver steps** are highest in the first interval, when the radicals spin up from zero, and jump at sunset and sunrise, when the light switches and the frozen rate constants change most from one interval to the next. The slower night chemistry needs the fewest.
> - **Hand-offs:** H₂SO₄ (from SO₂ + OH) reaches the particles mostly by day, nitrate (from N₂O₅ uptake) mostly at night. Each is passed on as a constant source over the interval; that is the splitting approximation.

> <font color="#A47F52"><b>TRY THIS</b></font> &nbsp; Edit `s` in the set-up cell, then re-run the cells from there:
>
> - `s = at.scenario_from("level3_day_night", numerics={"coupling_dt_s": 300.0})`: a three times shorter coupling interval (allowed: 300, 600, 900 s; each new value compiles the aerosol step once, and the loop then takes about twice as long). How much do nitrate and SOA change?
> - `s = at.scenario_from("level3_day_night", hold_constant=[])`: NO₂ is no longer reset in step 5. What happens to night-time N₂O₅ and nitrate (ASSUMPTIONS A14)?
> - `s = at.scenario_from("level3_day_night", gamma_n2o5=0.0)`: no uptake. Where does the nitrogen go instead?
> - `s = at.scenario_from("level3_day_night", cstar_table=None)`: every product gets the composition-only $c^*$ of Li et al. (2016) instead of the table (Section 3). How much SOA is there now, and why does the nitrate change too?

## Questions to think about

> 1. With the sun low (SZA 75–80°), TUV-x gives 2–4 times the MCM value of J(O¹D). Which Level 3 outputs would you expect to depend most on that choice, and at what time of day?
> 2. The model holds $J$ and the RO₂ sum fixed for a whole 900 s interval. When during the day is that least accurate, and which species would feel it most?
> 3. Roughly how many steps would an explicit solver need for the 24 h Level 3 run? Why does an implicit solver not have this problem?
> 4. Why does the model end below the Pankow equilibrium for 80 nm seeds, and which volatility classes are affected most?
> 5. For the Level 3 seed and γ = 0.02, what N₂O₅ lifetime do you get against uptake? How does that compare with the fixed chamber sink of 4×10⁻⁴ s⁻¹ it replaced?
> 6. Operator splitting hands H₂SO₄ and nitrate to the particles as constant sources over each interval. When would that be a poor approximation?
> 7. In notebook 2 (experiment 6c), 2 ppb of limonene gives an SOA yield (SOA formed per limonene reacted) of about 0.02 and 10 ppb about 0.16. How much of that difference can $C_{OA}$ in $F_i = (1 + c^*_i/C_{OA})^{-1}$ explain, and what else changes when there is less limonene? Hint: compare OH and the share of peroxy radicals that react with HO₂ (notebook 2, Section 6a).

## Going further

- **The code.** Read the code you used: the cell below prints links to the exact model version this notebook
  runs. `box/driver.py` is the full version of the loop in Section 5.
- **Documentation.** Model documentation in the tomas-jax repository: `docs/aaar/gaschem.md` (mechanism, rates,
  solver benchmark, photolysis), `coupling.md`, `soa_partitioning.md`, `heterogeneous.md`, and
  `ASSUMPTIONS.md`, which lists every simplification (A1–A14, AC1–AC2).
- **Photolysis modes.** Compare photolysis modes in a full run:
  `at.run("level3_day_night", photolysis={"mode": "mcm"})` against the TUV-x default.
- **c\* values.** c* is the most uncertain input. `tomas_jax.gaschem.volatility.load_cstar_table` reads the
  shipped table (`tomas_jax/gaschem/mechanisms/mcm_limonene_cstar_298K.csv`) as {species: log10 c*}, and
  `assign_classes(mech_def, table)` lumps it into classes; see how the class sizes change when you raise
  the c* of the hydroperoxides, which make up most of the SOA. In a full run,
  `at.run("level2_limonene_soa", cstar_ugm3={"LIMALOOH": 1e3})` changes one species and keeps the table's
  values for all the others.

In [ ]:
source = f"{config.GITHUB_ORG_URL}/TOMAS-JAX/blob/{config.TOMAS_JAX_COMMIT}"
for path in ("tomas_jax/gaschem/photolysis.py", "tomas_jax/gaschem/mechanism.py", "tomas_jax/gaschem/solver.py",
             "tomas_jax/physics/soa_partitioning.py", "tomas_jax/physics/heterogeneous.py",
             "tomas_jax/box/driver.py", "docs/aaar/coupling.md", "ASSUMPTIONS.md"):
    print(f"{source}/{path}")

### References

<details>
<summary>15 references (click to show)</summary>

- Saunders et al. (2003) and Jenkin et al. (1997, 2015): MCM. Saunders 2003: [doi:10.5194/acp-3-161-2003](https://doi.org/10.5194/acp-3-161-2003), [PDF](https://acp.copernicus.org/articles/3/161/2003/acp-3-161-2003.pdf); Jenkin 1997: [doi:10.1016/S1352-2310(96)00105-7](https://doi.org/10.1016/S1352-2310%2896%2900105-7); Jenkin 2015: [doi:10.5194/acp-15-11433-2015](https://doi.org/10.5194/acp-15-11433-2015), [PDF](https://acp.copernicus.org/articles/15/11433/2015/acp-15-11433-2015.pdf)
- Pankow (1994), *Atmos. Environ.* 28, 185–188. [doi:10.1016/1352-2310(94)90093-0](https://doi.org/10.1016/1352-2310%2894%2990093-0)
- Donahue et al. (2006), *Environ. Sci. Technol.* 40, 2635–2643. [doi:10.1021/es052297c](https://doi.org/10.1021/es052297c)
- Zaveri et al. (2014), *Atmos. Chem. Phys.* 14, 5153–5181. [doi:10.5194/acp-14-5153-2014](https://doi.org/10.5194/acp-14-5153-2014), [PDF](https://acp.copernicus.org/articles/14/5153/2014/acp-14-5153-2014.pdf)
- Epstein et al. (2010), *Environ. Sci. Technol.* 44, 743–748. [doi:10.1021/es902497z](https://doi.org/10.1021/es902497z)
- Nannoolal et al. (2008), *Fluid Phase Equilib.* 269, 117–133 (the vapour pressures behind the c* table). [doi:10.1016/j.fluid.2008.04.020](https://doi.org/10.1016/j.fluid.2008.04.020)
- O'Meara et al. (2021), *Geosci. Model Dev.* 14, 675–702 (PyCHAM). [doi:10.5194/gmd-14-675-2021](https://doi.org/10.5194/gmd-14-675-2021), [PDF](https://gmd.copernicus.org/articles/14/675/2021/gmd-14-675-2021.pdf)
- Li et al. (2016), *Atmos. Chem. Phys.* 16, 3327–3344. [doi:10.5194/acp-16-3327-2016](https://doi.org/10.5194/acp-16-3327-2016), [PDF](https://acp.copernicus.org/articles/16/3327/2016/acp-16-3327-2016.pdf)
- Fuchs and Sutugin (1971), *Topics in Current Aerosol Research*. [doi:10.1016/B978-0-08-016674-2.50006-6](https://doi.org/10.1016/B978-0-08-016674-2.50006-6)
- Bertram and Thornton (2009), *Atmos. Chem. Phys.* 9, 8351–8363. [doi:10.5194/acp-9-8351-2009](https://doi.org/10.5194/acp-9-8351-2009), [PDF](https://acp.copernicus.org/articles/9/8351/2009/acp-9-8351-2009.pdf)
- Kidger (2021), *On Neural Differential Equations* (diffrax). [arXiv:2202.02435](https://arxiv.org/abs/2202.02435), [PDF](https://arxiv.org/pdf/2202.02435)
- Kværnø (2004), *BIT Numer. Math.* 44, 489–502 (the Kvaerno solvers). [doi:10.1023/B:BITN.0000046811.70614.38](https://doi.org/10.1023/B:BITN.0000046811.70614.38)
- Hairer and Wanner (1996), *Solving Ordinary Differential Equations II: Stiff and Differential-Algebraic Problems*. [doi:10.1007/978-3-642-05221-7](https://doi.org/10.1007/978-3-642-05221-7)

</details>

---

<a href="https://www.reflective.org"><img src="https://raw.githubusercontent.com/reflective-org/AAAR-2026-Aerosol-Tutorial/9feba6889d498d8c754a6783e5460f46e054d60f/assets/supported-by-reflective.png" width="294" alt="Supported by Reflective"></a>